In [ ]:
import numpy as np
import pyxdf
import pandas as pd
import mne
import matplotlib.pyplot as plt
import os

In [2]:
from pathlib import Path
xdf_path = Path(r"C:\Users\jshin\OW_closedloopLIFU\xdf_data\sub-dave_run_4\ses-1\eeg\sub-dave_run_4_ses-1_task-dave_run_4_run-001_eeg.xdf")
data, header = pyxdf.load_xdf(str(xdf_path))
for i, stream in enumerate(data):
    print(f"Stream {i}: {stream['info']['name'][0]}")

In [3]:
stream = data[2]
df= []
df = pd.DataFrame(stream['time_series'])
df = df.rename(columns={i: f"Ch{i}" for i in range(df.shape[1])})

# Add timestamp column
df['Timestamp'] = stream['time_stamps']

# Move Timestamp to be the first column after the index
cols = ['Timestamp'] + [col for col in df.columns if col != 'Timestamp']
df = df[cols]
eeg_raw = df[['Ch0', 'Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6', 'Ch12','Timestamp']]
eeg_raw

In [4]:
EEG_LIFU_events = data[0]

# Create DataFrame from time_series
markers = pd.DataFrame(EEG_LIFU_events['time_series'])

# Rename the first column to 'markers'
markers.rename(columns={0: 'markers'}, inplace=True)

# Add timestamp column
markers['Timestamp'] = EEG_LIFU_events['time_stamps']

# Move Timestamp to be the first column after the index
cols = ['Timestamp'] + [col for col in markers.columns if col != 'Timestamp']
markers = markers[cols]
markers

In [5]:
lifu_on = markers[markers['markers'] == 'LIFU_ON']
lifu_on = np.array(lifu_on['Timestamp'])
lifu_on

In [6]:
pre = 2
post = 7
fs = 250  # sampling rate in Hz -- adjust if this isn't right for your data
pre_samples = int(pre * fs)    # 500
post_samples = int(post * fs)  # 1750
window_len = pre_samples + post_samples  # 2250

raw = eeg_raw.reset_index(drop=True)
timestamps = raw["Timestamp"].values

raw_windows = []
for idx, event in enumerate(lifu_on):
    center_idx = np.abs(timestamps - event).argmin()  # closest sample to onset
    start_idx = center_idx - pre_samples
    end_idx = start_idx + window_len

    if start_idx < 0 or end_idx > len(raw):
        print(f"Skipping event {idx}: window out of bounds ({start_idx}:{end_idx})")
        continue

    window_df = raw.iloc[start_idx:end_idx].copy()
    window_df["t_rel"] = np.arange(-pre_samples, post_samples) / fs
    window_df["idx"] = idx
    raw_windows.append(window_df)

raw_windows[0]

In [7]:
pre = -2
channel_8 = np.array(raw_windows[5]['Ch12'])
n = len(channel_8)   
t = np.linspace(pre, post, n)
plt.plot(t, channel_8)

In [8]:
import numpy as np
import mne

sfreq = 250

all_epochs = []

for idx in range(len(raw_windows)):
    df = raw_windows[idx].copy()
    df.columns = df.columns.str.strip()

    # Drop non‑EEG columns
    df = df.drop(['Timestamp', 't_rel', 'idx', 'Ch12'], axis=1)

    # Convert to numpy (n_channels × n_times)
    data = df.to_numpy().T

    all_epochs.append(data)

# Stack into shape (n_epochs, n_channels, n_times)
all_epochs = np.stack(all_epochs, axis=0)

# Channel names and types
ch_names = list(df.columns)
ch_types = ['eeg'] * len(ch_names)

# Create info
info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)

# Create MNE Epochs object
epochs = mne.EpochsArray(all_epochs, info)


In [9]:
from mne.time_frequency import stft
import numpy as np
import matplotlib.pyplot as plt

sfreq = epochs.info['sfreq']
tmin = -2
tmax = 7

data_all = epochs.get_data()   # shape: (n_epochs, n_channels, n_times)

all_spectrograms = []
all_times = []
all_freqs = []

for idx in range(len(data_all)):

    # ep shape: (n_channels, n_times)
    ep = data_all[idx].astype(float)

    data = ep #* 1e-6

    # stft params
    n_fft = 256
    step = 64
    
    Zxx = stft(data, wsize=n_fft, tstep=step)
    power = np.abs(Zxx)**2 / (n_fft**2)

    # time axis
    times = np.linspace(tmin, tmax, power.shape[-1])

    # frequency axis 
    freqs = np.linspace(0, sfreq/2, power.shape[1])

    # select all freqs
    mask = (freqs >= 1) 
    freqs_1_40 = freqs[mask]
    power_1_40 = power[:, mask, :]

    # one graph per epoch
    psd_avg = power_1_40.mean(axis=0)

    # db for better visualization
    psd_db = 10 * np.log10(psd_avg + 1e-20)

    vmin = np.percentile(psd_db, 5)
    vmax = np.percentile(psd_db, 95)

    all_spectrograms.append(psd_db)
    all_times.append(times)
    all_freqs.append(freqs_1_40)

    # --- Plot ---
    plt.figure(figsize=(10,6))
    plt.imshow(
        psd_db,
        aspect='auto',
        origin='lower',
        extent=[times[0], times[-1], freqs_1_40[0], freqs_1_40[-1]],
        vmin=vmin,
        vmax=vmax,
        cmap='viridis'
    )
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")
    plt.title(f"Epoch {idx} — Sonicating Time‑Resolved PSD")
    plt.colorbar(label="Power (dB)")
    plt.show()
